In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 4.2 MB/s eta 0:00:00
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
# ================================================================
# Schritt 1: Dateien und vorhandene Funktionen prüfen
# ================================================================
import os
import sys

# Ordner, in dem das Notebook ausgeführt wird
basis = os.getcwd()
print("Notebook-Ordner:", basis)

# Alle Dateien im aktuellen Ordner einlesen
dateien = sorted(os.listdir(basis))

# Benötigte Python-Dateien prüfen
for name in ["bom_core.py", "layer2_profile.py"]:
    vorhanden = name in dateien
    print(f"{name}: {'GEFUNDEN' if vorhanden else 'FEHLT'}")
    assert vorhanden, f"{name} wurde im Notebook-Ordner nicht gefunden"

# Gezielt nach der korrigierten BOM suchen
bom_kandidaten = [
    name for name in dateien
    if name.lower().endswith("_korrigiert.xlsx")
]

print("Korrigierte BOM-Dateien gefunden:", len(bom_kandidaten))
for name in bom_kandidaten:
    print(" -", name)

assert len(bom_kandidaten) == 1, (
    "Erwartet wird genau eine Excel-Datei mit dem Ende "
    "'_korrigiert.xlsx'."
)

arbeitsdatei = os.path.join(basis, bom_kandidaten[0])

# Aktuellen Ordner für Python-Imports verfügbar machen
if basis not in sys.path:
    sys.path.insert(0, basis)

# Bestehenden Projektcode importieren
import bom_core as bc
import layer2_profile as L2

# Prüfen, ob die benötigten Funktionen wirklich vorhanden sind
assert callable(getattr(bc, "clean_bom", None)), (
    "bom_core.py enthält keine aufrufbare Funktion clean_bom"
)
assert callable(getattr(L2, "baue", None)), (
    "layer2_profile.py enthält keine aufrufbare Funktion baue"
)

print("\nImport erfolgreich.")
print("clean_bom() vorhanden:", callable(bc.clean_bom))
print("L2.baue() vorhanden:", callable(L2.baue))
print("BOM-Pfad vorbereitet:", arbeitsdatei)
print("\nEs wurde noch keine BOM bereinigt und kein Modell gestartet.")

Notebook-Ordner: /Workspace/Users/khalil-said.albert@de.abb.com
bom_core.py: GEFUNDEN
layer2_profile.py: GEFUNDEN
Korrigierte BOM-Dateien gefunden: 1
 - 191007_Stuecklisten.aufgeloest (002)_korrigiert.xlsx

Import erfolgreich.
clean_bom() vorhanden: True
L2.baue() vorhanden: True
BOM-Pfad vorbereitet: /Workspace/Users/khalil-said.albert@de.abb.com/191007_Stuecklisten.aufgeloest (002)_korrigiert.xlsx

Es wurde noch keine BOM bereinigt und kein Modell gestartet.


In [0]:
# ================================================================
# Schritt 2: BOM mit dem bestehenden Cleaner bereinigen
# ================================================================
import time
from collections import defaultdict

start = time.time()

# Hier passiert die tatsächliche Bereinigung der Excel-BOM
sauber, protokoll = bc.clean_bom(arbeitsdatei)

dauer = time.time() - start

print(f"Cleaner beendet nach {dauer:.1f} Sekunden")
print("Typ von sauber:", type(sauber).__name__)
print("Bereinigte Positionszeilen:", len(sauber))

# Nur neutrale Kontrollzahlen berechnen – keine Namen ausgeben
produkte = {
    str(zeile["erzeugnis"])
    for zeile in sauber
}

komponenten = {
    str(zeile["komponente"])
    for zeile in sauber
}

bloecke_je_produkt = defaultdict(set)

for zeile in sauber:
    bloecke_je_produkt[str(zeile["erzeugnis"])].add(zeile["block"])

anzahl_bloecke = sum(
    len(bloecke)
    for bloecke in bloecke_je_produkt.values()
)

produkte_mit_mehreren_bloecken = sum(
    len(bloecke) > 1
    for bloecke in bloecke_je_produkt.values()
)

print("Produkte:", len(produkte))
print("Produktblöcke:", anzahl_bloecke)
print("Produkte mit mehreren Blöcken:", produkte_mit_mehreren_bloecken)
print("Unterschiedliche Komponenten:", len(komponenten))

# Erwartete Werte des bestehenden, bereits bekannten Clean-Stands
erwartet = {
    "Positionszeilen": (len(sauber), 47390),
    "Produkte": (len(produkte), 1692),
    "Produktblöcke": (anzahl_bloecke, 1822),
    "Produkte mit mehreren Blöcken": (
        produkte_mit_mehreren_bloecken, 130
    ),
    "Unterschiedliche Komponenten": (len(komponenten), 157),
}

print("\nKontrollvergleich:")
alles_ok = True

for kennzahl, (ist, soll) in erwartet.items():
    status = "OK" if ist == soll else "ABWEICHUNG"
    alles_ok = alles_ok and ist == soll
    print(f"{kennzahl}: {ist} | erwartet {soll} | {status}")

print("\nGesamtergebnis:", "CLEAN-STAND REPRODUZIERT" if alles_ok
      else "STOPP – ABWEICHUNG UNTERSUCHEN")

Cleaner beendet nach 75.5 Sekunden
Typ von sauber: list
Bereinigte Positionszeilen: 47390
Produkte: 1692
Produktblöcke: 1822
Produkte mit mehreren Blöcken: 130
Unterschiedliche Komponenten: 157

Kontrollvergleich:
Positionszeilen: 47390 | erwartet 47390 | OK
Produkte: 1692 | erwartet 1692 | OK
Produktblöcke: 1822 | erwartet 1822 | OK
Produkte mit mehreren Blöcken: 130 | erwartet 130 | OK
Unterschiedliche Komponenten: 157 | erwartet 157 | OK

Gesamtergebnis: CLEAN-STAND REPRODUZIERT


In [0]:
# ================================================================
# Schritt 3: Produkt-Komponenten-Daten aus der sauberen BOM bilden
# ================================================================
import pandas as pd

# Jede bereinigte Stücklistenposition wird zunächst übernommen.
# Hauptgruppe und kleinere Gruppe werden mit den bestehenden
# freigegebenen Funktionen bestimmt.
produkt_komponente = pd.DataFrame([
    {
        "produkt": str(zeile["erzeugnis"]),
        "hauptgruppe": L2.hauptgruppe_von(zeile["erzeugnis"]),
        "kleinere_gruppe": L2.typ_von(zeile["erzeugnis_txt"]),
        "komponente": str(zeile["komponente"]),
    }
    for zeile in sauber
])

print("Positionszeilen vor Zusammenfassung:",
      len(produkt_komponente))

# Prüfen, ob jedes Produkt eindeutig genau einer Hauptgruppe
# und genau einer kleineren Gruppe zugeordnet wird.
zuordnungen = (
    produkt_komponente
    .groupby("produkt")[["hauptgruppe", "kleinere_gruppe"]]
    .nunique()
)

mehrdeutige_hauptgruppen = int(
    (zuordnungen["hauptgruppe"] > 1).sum()
)

mehrdeutige_kleinere_gruppen = int(
    (zuordnungen["kleinere_gruppe"] > 1).sum()
)

print("Produkte mit mehreren Hauptgruppen:",
      mehrdeutige_hauptgruppen)
print("Produkte mit mehreren kleineren Gruppen:",
      mehrdeutige_kleinere_gruppen)

assert mehrdeutige_hauptgruppen == 0
assert mehrdeutige_kleinere_gruppen == 0

# Dieselbe Komponente darf pro Produkt nur einmal vorkommen.
# Mehrere Blöcke oder Positionszeilen erhöhen ihr Gewicht nicht.
produkt_komponente = (
    produkt_komponente
    .drop_duplicates(["produkt", "komponente"])
    .reset_index(drop=True)
)

# Nur neutrale Kontrollzahlen ausgeben
anzahl_produkte = produkt_komponente["produkt"].nunique()
anzahl_hauptgruppen = produkt_komponente["hauptgruppe"].nunique()
anzahl_kleinere_gruppen = (
    produkt_komponente[
        ["hauptgruppe", "kleinere_gruppe"]
    ]
    .drop_duplicates()
    .shape[0]
)
anzahl_komponenten = produkt_komponente["komponente"].nunique()

print("\nNach einmaliger Zählung je Produkt und Komponente:")
print("Produkt-Komponenten-Kombinationen:",
      len(produkt_komponente))
print("Produkte:", anzahl_produkte)
print("Hauptgruppen:", anzahl_hauptgruppen)
print("Kleinere Gruppen:", anzahl_kleinere_gruppen)
print("Unterschiedliche Komponenten:", anzahl_komponenten)

print("\nKontrollvergleich:")
print("Produkte:", "OK" if anzahl_produkte == 1692 else "ABWEICHUNG")
print("Hauptgruppen:", "OK" if anzahl_hauptgruppen == 4 else "ABWEICHUNG")
print("Kleinere Gruppen:",
      "OK" if anzahl_kleinere_gruppen == 37 else "ABWEICHUNG")
print("Komponenten:",
      "OK" if anzahl_komponenten == 157 else "ABWEICHUNG")

Positionszeilen vor Zusammenfassung: 47390
Produkte mit mehreren Hauptgruppen: 0
Produkte mit mehreren kleineren Gruppen: 0

Nach einmaliger Zählung je Produkt und Komponente:
Produkt-Komponenten-Kombinationen: 38654
Produkte: 1692
Hauptgruppen: 4
Kleinere Gruppen: 37
Unterschiedliche Komponenten: 157

Kontrollvergleich:
Produkte: OK
Hauptgruppen: OK
Kleinere Gruppen: OK
Komponenten: OK


In [0]:
# ================================================================
# Schritt 4: Vollständige Produkt-Komponenten-Matrix
# ================================================================

# Eindeutige Gruppenzuordnung jedes Produkts
produkt_info_alle = (
    produkt_komponente[
        ["produkt", "hauptgruppe", "kleinere_gruppe"]
    ]
    .drop_duplicates()
    .set_index("produkt")
)

assert not produkt_info_alle.index.duplicated().any()

# Zeile = Produkt
# Spalte = Komponente
# 1 = im Produkt vorhanden
# 0 = im Produkt nicht vorhanden
X_produkte_alle = pd.crosstab(
    index=produkt_komponente["produkt"],
    columns=produkt_komponente["komponente"]
)

X_produkte_alle = (X_produkte_alle > 0).astype("int8")

# Gruppenzuordnungen in dieselbe Reihenfolge wie die Matrix bringen
produkt_info_alle = produkt_info_alle.loc[X_produkte_alle.index]

belegte_zellen = int(X_produkte_alle.values.sum())
alle_zellen = int(
    X_produkte_alle.shape[0] * X_produkte_alle.shape[1]
)
anteil_vorhanden = belegte_zellen / alle_zellen

anzahl_kleinere_gruppen = (
    produkt_info_alle[
        ["hauptgruppe", "kleinere_gruppe"]
    ]
    .drop_duplicates()
    .shape[0]
)

print("Vollständige Modelleingabe:")
print("Produkte:", X_produkte_alle.shape[0])
print("Komponenten:", X_produkte_alle.shape[1])
print("Matrixform:", X_produkte_alle.shape)
print("Kleinere Gruppen:", anzahl_kleinere_gruppen)
print("Vorhandene Produkt-Komponenten-Verbindungen:",
      belegte_zellen)
print(f"Anteil vorhandener Verbindungen: {anteil_vorhanden:.3%}")

print("\nKontrollvergleich:")
print("Produkte:",
      "OK" if X_produkte_alle.shape[0] == 1692 else "ABWEICHUNG")
print("Komponenten:",
      "OK" if X_produkte_alle.shape[1] == 157 else "ABWEICHUNG")
print("Kleinere Gruppen:",
      "OK" if anzahl_kleinere_gruppen == 37 else "ABWEICHUNG")
print("Verbindungen:",
      "OK" if belegte_zellen == 38654 else "ABWEICHUNG")

print("\nKeine Produkte und keine kleineren Gruppen ausgeschlossen.")


Vollständige Modelleingabe:
Produkte: 1692
Komponenten: 157
Matrixform: (1692, 157)
Kleinere Gruppen: 37
Vorhandene Produkt-Komponenten-Verbindungen: 38654
Anteil vorhandener Verbindungen: 14.551%

Kontrollvergleich:
Produkte: OK
Komponenten: OK
Kleinere Gruppen: OK
Verbindungen: OK

Keine Produkte und keine kleineren Gruppen ausgeschlossen.


In [0]:
"""
PA1 – vollständige NMF-Prüfung für den Pilot einer Hauptgruppe.

Einbau im Databricks-Notebook:
    Diese Datei bzw. ihren Inhalt NACH Schritt 5 ausführen.

Benötigte Objekte im Arbeitsspeicher:
    X_ziel   : DataFrame, Zeilen = Produkte, Spalten = Komponenten
    ziel_info: DataFrame mit Spalte "kleinere_gruppe", Index = Produkte

Die kleineren Gruppen werden ausschließlich zur nachgelagerten Bewertung
verwendet. Sie gehen nicht in das NMF-Training ein.
"""

from pathlib import Path
from itertools import combinations
import hashlib
import shutil
import time
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import NMF
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics.pairwise import cosine_similarity


# ---------------------------------------------------------------------------
# 1. Einstellungen
# ---------------------------------------------------------------------------

K_WERTE = [2, 3, 4, 5, 6, 8, 10, 12, 15]
SEEDS = [20260908, 20260909, 20260910, 20260911, 20260912]
MAX_ITER = 3000
TOL = 1e-4

# Fachlicher Anker bzw. mathematisch auffällige Paare. Diese Namen werden
# NICHT trainiert, sondern nur nach dem Training in der Rangliste gesucht.
ANKER_PAARE = {
    "B6 / B6S": ("B6", "B6S"),
    "VB6 / VB6A": ("VB6", "VB6A"),
    "VBC6 / VBC6A": ("VBC6", "VBC6A"),
}

MIN_VORKOMMEN_TEST = 5
MAX_ANTEIL_TEST = 0.95
REFERENZ_K = 6
REFERENZ_SEED = SEEDS[0]
EPS = 1e-12


# ---------------------------------------------------------------------------
# 2. Eingaben prüfen
# ---------------------------------------------------------------------------

assert "X_ziel" in globals(), "X_ziel fehlt – zuerst Notebook-Schritt 5 ausführen."
assert "ziel_info" in globals(), "ziel_info fehlt – zuerst Notebook-Schritt 5 ausführen."
assert isinstance(X_ziel, pd.DataFrame), "X_ziel muss ein pandas-DataFrame sein."
assert isinstance(ziel_info, pd.DataFrame), "ziel_info muss ein pandas-DataFrame sein."
assert "kleinere_gruppe" in ziel_info.columns
assert X_ziel.index.is_unique and X_ziel.columns.is_unique
assert set(X_ziel.index) <= set(ziel_info.index)

X_basis = X_ziel.astype(float).copy()
gruppen_labels = ziel_info.loc[X_basis.index, "kleinere_gruppe"].astype(str)

werte = np.unique(X_basis.to_numpy())
assert set(werte).issubset({0.0, 1.0}), f"X_ziel ist nicht binär: {werte[:10]}"
assert not X_basis.isna().any().any()
assert X_basis.shape[0] == gruppen_labels.shape[0]


# ---------------------------------------------------------------------------
# 3. Komponenten-Audit
# ---------------------------------------------------------------------------

def binaere_entropie(p):
    p = np.asarray(p, dtype=float)
    q = 1.0 - p
    out = np.zeros_like(p)
    maske_p = p > 0
    maske_q = q > 0
    out[maske_p] -= p[maske_p] * np.log2(p[maske_p])
    out[maske_q] -= q[maske_q] * np.log2(q[maske_q])
    return out


def signatur_spalte(spalte):
    gepackt = np.packbits(np.asarray(spalte, dtype=np.uint8)).tobytes()
    return hashlib.sha256(gepackt).hexdigest()[:16]


n_produkte = X_basis.shape[0]
vorkommen = X_basis.sum(axis=0).astype(int)
anteil = vorkommen / n_produkte

gruppen_praevalenz = (
    X_basis.assign(_kleinere_gruppe=gruppen_labels)
    .groupby("_kleinere_gruppe")
    .mean()
)

audit = pd.DataFrame({
    "komponente": X_basis.columns.astype(str),
    "produkte_mit_komponente": vorkommen.to_numpy(),
    "anteil_produkte": anteil.to_numpy(),
    "varianz": (anteil * (1.0 - anteil)).to_numpy(),
    "entropie_bit": binaere_entropie(anteil.to_numpy()),
    "min_anteil_kleinere_gruppe": gruppen_praevalenz.min(axis=0).to_numpy(),
    "max_anteil_kleinere_gruppe": gruppen_praevalenz.max(axis=0).to_numpy(),
    "spannweite_kleinere_gruppen": (
        gruppen_praevalenz.max(axis=0) - gruppen_praevalenz.min(axis=0)
    ).to_numpy(),
    "praesenz_signatur": [
        signatur_spalte(X_basis.iloc[:, j].to_numpy())
        for j in range(X_basis.shape[1])
    ],
})

signatur_groesse = audit["praesenz_signatur"].value_counts()
audit["identisches_profil_N"] = audit["praesenz_signatur"].map(signatur_groesse)

bedingungen = [
    audit["produkte_mit_komponente"] == n_produkte,
    audit["produkte_mit_komponente"] < MIN_VORKOMMEN_TEST,
    audit["anteil_produkte"] >= MAX_ANTEIL_TEST,
]
klassen = [
    "SICHER_ENTFERNBAR_KONSTANT_1",
    "SELTEN_NUR_TESTWEISE_ENTFERNEN",
    "SEHR_HAEUFIG_NUR_TESTWEISE_ENTFERNEN",
]
audit["audit_klasse"] = np.select(bedingungen, klassen, default="BEHALTEN")

audit["begruendung"] = np.select(
    bedingungen,
    [
        "In jedem Produkt vorhanden; besitzt innerhalb dieser Hauptgruppe keine Varianz.",
        f"In weniger als {MIN_VORKOMMEN_TEST} Produkten; kann ein legitimer Einzelfall oder Rauschen sein.",
        f"In mindestens {MAX_ANTEIL_TEST:.0%} der Produkte; kann den Frobenius-Fehler dominieren.",
    ],
    default="Besitzt für den Pilot grundsätzlich unterscheidende Information.",
)

nmf_komponenten_audit = audit.sort_values(
    ["audit_klasse", "produkte_mit_komponente", "komponente"],
    ascending=[True, False, True],
).reset_index(drop=True)

nmf_identische_profile = (
    audit[audit["identisches_profil_N"] > 1]
    .groupby("praesenz_signatur")
    .agg(
        komponenten_N=("komponente", "size"),
        komponenten=("komponente", lambda s: " | ".join(sorted(s))),
        produkte_mit_profil=("produkte_mit_komponente", "first"),
        anteil_produkte=("anteil_produkte", "first"),
    )
    .sort_values(["komponenten_N", "produkte_mit_profil"], ascending=False)
    .reset_index()
)


# ---------------------------------------------------------------------------
# 4. Datenvarianten – nur Konstanten werden als sicher entfernbar bezeichnet
# ---------------------------------------------------------------------------

masken = {
    "RAW": np.ones(X_basis.shape[1], dtype=bool),
    "OHNE_KONSTANTE": (vorkommen < n_produkte).to_numpy(),
    "MINDESTENS_5": (
        (vorkommen >= MIN_VORKOMMEN_TEST) & (vorkommen < n_produkte)
    ).to_numpy(),
    "MINDESTENS_5_MAXIMAL_95_PROZENT": (
        (vorkommen >= MIN_VORKOMMEN_TEST)
        & (anteil < MAX_ANTEIL_TEST)
    ).to_numpy(),
}

varianten = {
    name: X_basis.loc[:, X_basis.columns[maske]].copy()
    for name, maske in masken.items()
}

nmf_varianten_uebersicht = pd.DataFrame([
    {
        "variante": name,
        "produkte": X.shape[0],
        "komponenten": X.shape[1],
        "entfernte_komponenten": X_basis.shape[1] - X.shape[1],
        "einsen": int(X.to_numpy().sum()),
        "dichte": float(X.to_numpy().mean()),
    }
    for name, X in varianten.items()
])


# ---------------------------------------------------------------------------
# 5. Skalenfeste W-Profile und Kennzahlen
# ---------------------------------------------------------------------------

def skalenfeste_w_profile(W, H):
    """Macht W-Profile invariant gegenüber W*D und D^-1*H.

    Jedes H-Muster wird gedanklich auf Zeilensumme 1 normiert. Seine entfernte
    Masse wird in die zugehörige W-Spalte übertragen. Erst danach werden die
    Produktprofile auf Summe 1 normiert.
    """
    muster_masse = H.sum(axis=1)
    W_beitrag = W * muster_masse[None, :]
    zeilensumme = W_beitrag.sum(axis=1, keepdims=True)
    return np.divide(
        W_beitrag,
        zeilensumme,
        out=np.zeros_like(W_beitrag),
        where=zeilensumme > EPS,
    )


def effektive_komponenten(H):
    zeilensummen = H.sum(axis=1, keepdims=True)
    anteile = np.divide(H, zeilensummen, out=np.zeros_like(H), where=zeilensummen > EPS)
    quadratsummen = np.sum(anteile ** 2, axis=1)
    return np.divide(1.0, quadratsummen, out=np.zeros_like(quadratsummen), where=quadratsummen > EPS)


def gruppen_auswertung(W_rel, labels):
    W_df = pd.DataFrame(W_rel, index=labels.index)
    gruppenmittel = W_df.assign(_gruppe=labels).groupby("_gruppe").mean()
    gruppengroessen = labels.value_counts()

    # Produktähnlichkeit zum Mittel seiner kleineren Gruppe.
    koh = []
    for gruppe, indizes in labels.groupby(labels).groups.items():
        if len(indizes) < 2:
            continue
        produktprofile = W_df.loc[indizes].to_numpy()
        zentrum = gruppenmittel.loc[[gruppe]].to_numpy()
        koh.extend(cosine_similarity(produktprofile, zentrum).ravel().tolist())

    namen = list(gruppenmittel.index)
    S = cosine_similarity(gruppenmittel.to_numpy())
    paare = []
    for i, j in combinations(range(len(namen)), 2):
        paare.append({
            "gruppe_a": namen[i],
            "gruppe_b": namen[j],
            "produkte_a": int(gruppengroessen[namen[i]]),
            "produkte_b": int(gruppengroessen[namen[j]]),
            "aehnlichkeit": float(S[i, j]),
        })
    paare = pd.DataFrame(paare).sort_values("aehnlichkeit", ascending=False).reset_index(drop=True)
    paare["rang"] = np.arange(1, len(paare) + 1)

    zwischen = paare["aehnlichkeit"].to_numpy() if len(paare) else np.array([np.nan])
    return {
        "gruppenmittel": gruppenmittel,
        "paare": paare,
        "kohesion_mittel": float(np.mean(koh)) if koh else np.nan,
        "kohesion_median": float(np.median(koh)) if koh else np.nan,
        "zwischen_median": float(np.nanmedian(zwischen)),
        "zwischen_q90": float(np.nanquantile(zwischen, 0.90)),
    }


def anker_aus_paaren(paare):
    ausgabe = []
    for anker_name, (a, b) in ANKER_PAARE.items():
        treffer = paare[
            paare.apply(
                lambda z: {z["gruppe_a"], z["gruppe_b"]} == {a, b},
                axis=1,
            )
        ]
        if treffer.empty:
            ausgabe.append({
                "anker": anker_name,
                "vorhanden": False,
                "aehnlichkeit": np.nan,
                "rang": np.nan,
                "paare_gesamt": len(paare),
            })
        else:
            z = treffer.iloc[0]
            ausgabe.append({
                "anker": anker_name,
                "vorhanden": True,
                "aehnlichkeit": float(z["aehnlichkeit"]),
                "rang": int(z["rang"]),
                "paare_gesamt": len(paare),
            })
    return ausgabe


def h_stabilitaet(H_ref, H_test):
    """Ordnet permutierte Muster optimal zu und gibt mittlere Kosinusähnlichkeit zurück."""
    S = cosine_similarity(H_ref, H_test)
    zeilen, spalten = linear_sum_assignment(-S)
    return float(S[zeilen, spalten].mean()), list(zip(zeilen, spalten))


# ---------------------------------------------------------------------------
# 6. NMF über Varianten, k und echte Zufallsstarts rechnen
# ---------------------------------------------------------------------------

laufzeilen = []
paarzeilen = []
ankerzeilen = []
modelle_referenz = {}
H_fuer_stabilitaet = {}

for varianten_name, X_variante_df in varianten.items():
    Xv = X_variante_df.to_numpy(dtype=float)

    for k in K_WERTE:
        if k > min(Xv.shape):
            continue

        for seed in SEEDS:
            start = time.time()
            modell = NMF(
                n_components=k,
                init="nndsvdar",  # echte Seed-Variation; nndsvda wäre praktisch deterministisch
                solver="cd",
                beta_loss="frobenius",
                random_state=seed,
                max_iter=MAX_ITER,
                tol=TOL,
            )

            with warnings.catch_warnings(record=True) as gefangene_warnungen:
                warnings.simplefilter("always", ConvergenceWarning)
                W = modell.fit_transform(Xv)
                H = modell.components_

            konvergenzwarnung = any(
                issubclass(w.category, ConvergenceWarning)
                for w in gefangene_warnungen
            )

            rekonstruktion = W @ H
            rel_fehler = np.linalg.norm(Xv - rekonstruktion, ord="fro") / np.linalg.norm(Xv, ord="fro")
            W_rel = skalenfeste_w_profile(W, H)
            ga = gruppen_auswertung(W_rel, gruppen_labels)

            H_norm = np.linalg.norm(H, axis=1, keepdims=True)
            H_richtung = np.divide(H, H_norm, out=np.zeros_like(H), where=H_norm > EPS)
            S_H = cosine_similarity(H_richtung)
            ohne_diagonale = S_H[~np.eye(k, dtype=bool)]
            eff = effektive_komponenten(H)

            laufzeilen.append({
                "variante": varianten_name,
                "komponenten": Xv.shape[1],
                "k": k,
                "seed": seed,
                "init": "nndsvdar",
                "relativer_fehler": float(rel_fehler),
                "iterationen": int(modell.n_iter_),
                "konvergiert": bool(not konvergenzwarnung and modell.n_iter_ < MAX_ITER),
                "dauer_sekunden": time.time() - start,
                "leere_muster": int((H.sum(axis=1) <= EPS).sum()),
                "max_aehnlichkeit_zweier_H_muster": float(np.max(ohne_diagonale)) if len(ohne_diagonale) else np.nan,
                "effektive_komponenten_median": float(np.median(eff)),
                "kohesion_innerhalb_gruppe_mittel": ga["kohesion_mittel"],
                "kohesion_innerhalb_gruppe_median": ga["kohesion_median"],
                "aehnlichkeit_zwischen_gruppen_median": ga["zwischen_median"],
                "aehnlichkeit_zwischen_gruppen_q90": ga["zwischen_q90"],
            })

            for _, z in ga["paare"].iterrows():
                paarzeilen.append({
                    "variante": varianten_name,
                    "k": k,
                    "seed": seed,
                    **z.to_dict(),
                })

            for z in anker_aus_paaren(ga["paare"]):
                ankerzeilen.append({
                    "variante": varianten_name,
                    "k": k,
                    "seed": seed,
                    **z,
                })

            H_fuer_stabilitaet[(varianten_name, k, seed)] = H_richtung

            if seed == REFERENZ_SEED:
                modelle_referenz[(varianten_name, k)] = {
                    "modell": modell,
                    "W": W,
                    "H": H,
                    "W_rel": W_rel,
                    "komponenten": list(X_variante_df.columns),
                    "produkte": list(X_variante_df.index),
                }


nmf_einzel_laeufe = pd.DataFrame(laufzeilen)
nmf_gruppenpaare_alle_laeufe = pd.DataFrame(paarzeilen)
nmf_anker_alle_laeufe = pd.DataFrame(ankerzeilen)
nmf_eval_modelle = modelle_referenz


# ---------------------------------------------------------------------------
# 7. Stabilität passend zuordnen und Ergebnisse zusammenfassen
# ---------------------------------------------------------------------------

stabilitaetszeilen = []
for varianten_name in varianten:
    for k in K_WERTE:
        schluessel_ref = (varianten_name, k, REFERENZ_SEED)
        if schluessel_ref not in H_fuer_stabilitaet:
            continue
        H_ref = H_fuer_stabilitaet[schluessel_ref]
        for seed in SEEDS:
            schluessel = (varianten_name, k, seed)
            if schluessel not in H_fuer_stabilitaet:
                continue
            wert, _ = h_stabilitaet(H_ref, H_fuer_stabilitaet[schluessel])
            stabilitaetszeilen.append({
                "variante": varianten_name,
                "k": k,
                "seed": seed,
                "H_stabilitaet_zur_referenz": wert,
            })

nmf_H_stabilitaet = pd.DataFrame(stabilitaetszeilen)
nmf_einzel_laeufe = nmf_einzel_laeufe.merge(
    nmf_H_stabilitaet,
    on=["variante", "k", "seed"],
    how="left",
)

nmf_modellvergleich = (
    nmf_einzel_laeufe
    .groupby(["variante", "komponenten", "k"], as_index=False)
    .agg(
        laeufe=("seed", "size"),
        konvergenzquote=("konvergiert", "mean"),
        fehler_median=("relativer_fehler", "median"),
        fehler_std=("relativer_fehler", "std"),
        H_stabilitaet_median=("H_stabilitaet_zur_referenz", "median"),
        H_stabilitaet_min=("H_stabilitaet_zur_referenz", "min"),
        max_H_muster_aehnlichkeit_median=("max_aehnlichkeit_zweier_H_muster", "median"),
        leere_muster_max=("leere_muster", "max"),
        effektive_komponenten_median=("effektive_komponenten_median", "median"),
        gruppenkohesion_median=("kohesion_innerhalb_gruppe_median", "median"),
        gruppen_aehnlichkeit_median=("aehnlichkeit_zwischen_gruppen_median", "median"),
        gruppen_aehnlichkeit_q90=("aehnlichkeit_zwischen_gruppen_q90", "median"),
        dauer_median_s=("dauer_sekunden", "median"),
    )
    .sort_values(["variante", "k"])
    .reset_index(drop=True)
)

nmf_gruppenpaare_zusammenfassung = (
    nmf_gruppenpaare_alle_laeufe
    .groupby(["variante", "k", "gruppe_a", "gruppe_b"], as_index=False)
    .agg(
        produkte_a=("produkte_a", "first"),
        produkte_b=("produkte_b", "first"),
        aehnlichkeit_median=("aehnlichkeit", "median"),
        aehnlichkeit_std=("aehnlichkeit", "std"),
        rang_median=("rang", "median"),
        rang_min=("rang", "min"),
        rang_max=("rang", "max"),
    )
)

nmf_anker_zusammenfassung = (
    nmf_anker_alle_laeufe
    .groupby(["variante", "k", "anker"], as_index=False)
    .agg(
        vorhanden=("vorhanden", "all"),
        aehnlichkeit_median=("aehnlichkeit", "median"),
        aehnlichkeit_std=("aehnlichkeit", "std"),
        rang_median=("rang", "median"),
        rang_min=("rang", "min"),
        rang_max=("rang", "max"),
        paare_gesamt=("paare_gesamt", "first"),
    )
    .sort_values(["anker", "variante", "k"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# 8. Direkte, nicht gelernte Baseline auf denselben Datenvarianten
# ---------------------------------------------------------------------------

baseline_zeilen = []
for varianten_name, X_variante_df in varianten.items():
    gruppen_X = (
        X_variante_df.assign(_gruppe=gruppen_labels)
        .groupby("_gruppe")
        .mean()
    )
    namen = list(gruppen_X.index)
    A = gruppen_X.to_numpy(dtype=float)
    S = cosine_similarity(A)

    paare = []
    for i, j in combinations(range(len(namen)), 2):
        gewichteter_jaccard = np.minimum(A[i], A[j]).sum() / np.maximum(A[i], A[j]).sum()
        paare.append({
            "gruppe_a": namen[i],
            "gruppe_b": namen[j],
            "kosinus": float(S[i, j]),
            "gewichteter_jaccard": float(gewichteter_jaccard),
        })
    paare = pd.DataFrame(paare)
    paare["kosinus_rang"] = paare["kosinus"].rank(method="min", ascending=False).astype(int)
    paare["jaccard_rang"] = paare["gewichteter_jaccard"].rank(method="min", ascending=False).astype(int)
    paare.insert(0, "variante", varianten_name)
    baseline_zeilen.append(paare)

nmf_direkte_baseline = pd.concat(baseline_zeilen, ignore_index=True)


# ---------------------------------------------------------------------------
# 9. Top-Komponenten der Referenzmodelle sichtbar machen
# ---------------------------------------------------------------------------

top_zeilen = []
for (varianten_name, k), lauf in modelle_referenz.items():
    if k != REFERENZ_K:
        continue
    H = lauf["H"]
    komponenten = lauf["komponenten"]
    for i in range(k):
        masse = H[i].sum()
        ordnung = np.argsort(H[i])[::-1][:15]
        for rang, j in enumerate(ordnung, start=1):
            top_zeilen.append({
                "variante": varianten_name,
                "k": k,
                "muster": i + 1,
                "rang_im_muster": rang,
                "komponente": komponenten[j],
                "gewicht": float(H[i, j]),
                "anteil_am_muster": float(H[i, j] / masse) if masse > EPS else 0.0,
                "produkte_mit_komponente": int(X_basis[komponenten[j]].sum()),
                "anteil_produkte": float(X_basis[komponenten[j]].mean()),
            })

nmf_top_komponenten_k6 = pd.DataFrame(top_zeilen)


# ---------------------------------------------------------------------------
# 10. Kleine, lesbare Ausgabe und reproduzierbare Exporte
# ---------------------------------------------------------------------------

print("\n1) KOMPONENTEN-AUDIT")
print("-" * 70)
display(
    nmf_komponenten_audit["audit_klasse"]
    .value_counts()
    .rename_axis("audit_klasse")
    .reset_index(name="komponenten_N")
)
display(nmf_varianten_uebersicht)

print("\nSicher informationslos oder nur testweise auffällig:")
display(
    nmf_komponenten_audit[
        nmf_komponenten_audit["audit_klasse"] != "BEHALTEN"
    ][[
        "komponente", "produkte_mit_komponente", "anteil_produkte",
        "identisches_profil_N", "audit_klasse", "begruendung",
    ]]
)

print("\nKomponenten mit identischem Präsenzprofil (nicht automatisch löschen):")
display(nmf_identische_profile)

print("\n2) MODELLVERGLEICH")
print("-" * 70)
display(nmf_modellvergleich.round(4))

print("\n3) ANKERPAARE ÜBER k, FILTER UND ZUFALLSSTARTS")
print("-" * 70)
display(nmf_anker_zusammenfassung.round(4))

print("\n4) TOP-KOMPONENTEN DER k=6-REFERENZMODELLE")
print("-" * 70)
display(nmf_top_komponenten_k6.round(4))

print("\nINTERPRETATIONSREGELN")
print("- Fehler nur innerhalb derselben Datenvariante vergleichen.")
print("- H-Stabilität nahe 1 ist gut; niedrige Werte bedeuten startabhängige Muster.")
print("- Hohe Ähnlichkeit zweier H-Muster deutet auf redundante Muster hin.")
print("- Konstant-1-Komponenten dürfen entfernt werden; selten/häufig nur bei robusterem Ergebnis.")
print("- Anker-Ränge sollen über k, Seeds und vertretbare Filter stabil bleiben.")
print("- Hohe Gruppenähnlichkeit ist weiterhin keine Wahrscheinlichkeit fachlicher Richtigkeit.")

ausgabe_ordner = Path("nmf_evaluation_outputs")
ausgabe_ordner.mkdir(exist_ok=True)

exports = {
    "01_komponenten_audit.csv": nmf_komponenten_audit,
    "02_identische_praesenzprofile.csv": nmf_identische_profile,
    "03_datenvarianten.csv": nmf_varianten_uebersicht,
    "04_nmf_einzellaeufe.csv": nmf_einzel_laeufe,
    "05_nmf_modellvergleich.csv": nmf_modellvergleich,
    "06_nmf_anker.csv": nmf_anker_zusammenfassung,
    "07_nmf_gruppenpaare.csv": nmf_gruppenpaare_zusammenfassung,
    "08_direkte_baseline.csv": nmf_direkte_baseline,
    "09_top_komponenten_k6.csv": nmf_top_komponenten_k6,
}

for dateiname, tabelle in exports.items():
    tabelle.to_csv(ausgabe_ordner / dateiname, index=False)

zip_pfad = shutil.make_archive(
    str(ausgabe_ordner),
    "zip",
    root_dir=ausgabe_ordner,
)

print(f"\nFertig. {len(exports)} CSV-Dateien gespeichert unter: {ausgabe_ordner.resolve()}")
print(f"ZIP zum Weitergeben: {Path(zip_pfad).resolve()}")



1) KOMPONENTEN-AUDIT
----------------------------------------------------------------------


audit_klasse,komponenten_N
BEHALTEN,72
SELTEN_NUR_TESTWEISE_ENTFERNEN,20
SEHR_HAEUFIG_NUR_TESTWEISE_ENTFERNEN,11


variante,produkte,komponenten,entfernte_komponenten,einsen,dichte
RAW,772,103,0,17727,0.2229362643996177
OHNE_KONSTANTE,772,103,0,17727,0.2229362643996177
MINDESTENS_5,772,83,20,17684,0.2759847680878956
MINDESTENS_5_MAXIMAL_95_PROZENT,772,72,31,9382,0.16878957973517558



Sicher informationslos oder nur testweise auffällig:


komponente,produkte_mit_komponente,anteil_produkte,identisches_profil_N,audit_klasse,begruendung
GJL1203601P0002,766,0.9922279792746114,3,SEHR_HAEUFIG_NUR_TESTWEISE_ENTFERNEN,In mindestens 95% der Produkte; kann den Frobenius-Fehler dominieren.
GJL1206301P0005,766,0.9922279792746114,3,SEHR_HAEUFIG_NUR_TESTWEISE_ENTFERNEN,In mindestens 95% der Produkte; kann den Frobenius-Fehler dominieren.
GJL1206603P0010,766,0.9922279792746114,3,SEHR_HAEUFIG_NUR_TESTWEISE_ENTFERNEN,In mindestens 95% der Produkte; kann den Frobenius-Fehler dominieren.
GJL1201331R0007,756,0.9792746113989638,4,SEHR_HAEUFIG_NUR_TESTWEISE_ENTFERNEN,In mindestens 95% der Produkte; kann den Frobenius-Fehler dominieren.
GJL1201331R0008,756,0.9792746113989638,4,SEHR_HAEUFIG_NUR_TESTWEISE_ENTFERNEN,In mindestens 95% der Produkte; kann den Frobenius-Fehler dominieren.
GJL1201332R0002,756,0.9792746113989638,4,SEHR_HAEUFIG_NUR_TESTWEISE_ENTFERNEN,In mindestens 95% der Produkte; kann den Frobenius-Fehler dominieren.
GJL1206601P0015,756,0.9792746113989638,4,SEHR_HAEUFIG_NUR_TESTWEISE_ENTFERNEN,In mindestens 95% der Produkte; kann den Frobenius-Fehler dominieren.
GJL1201005R0004,746,0.966321243523316,3,SEHR_HAEUFIG_NUR_TESTWEISE_ENTFERNEN,In mindestens 95% der Produkte; kann den Frobenius-Fehler dominieren.
GJL1201008R0001,746,0.966321243523316,3,SEHR_HAEUFIG_NUR_TESTWEISE_ENTFERNEN,In mindestens 95% der Produkte; kann den Frobenius-Fehler dominieren.
GJL1206623P0015,746,0.966321243523316,3,SEHR_HAEUFIG_NUR_TESTWEISE_ENTFERNEN,In mindestens 95% der Produkte; kann den Frobenius-Fehler dominieren.


...


3) ANKERPAARE ÜBER k, FILTER UND ZUFALLSSTARTS
----------------------------------------------------------------------


variante,k,anker,vorhanden,aehnlichkeit_median,aehnlichkeit_std,rang_median,rang_min,rang_max,paare_gesamt
MINDESTENS_5,2,B6 / B6S,true,0.9866,0.0,17.0,17,17,78
MINDESTENS_5,3,B6 / B6S,true,0.9805,0.0,12.0,12,12,78
MINDESTENS_5,4,B6 / B6S,true,0.9806,0.0,10.0,10,10,78
MINDESTENS_5,5,B6 / B6S,true,0.8769,0.0,17.0,17,17,78
MINDESTENS_5,6,B6 / B6S,true,0.8788,0.0,16.0,16,16,78
MINDESTENS_5,8,B6 / B6S,true,0.905,0.0,5.0,5,5,78
MINDESTENS_5,10,B6 / B6S,true,0.901,0.0,5.0,5,5,78
MINDESTENS_5,12,B6 / B6S,true,0.9755,0.0,2.0,2,2,78
MINDESTENS_5,15,B6 / B6S,true,0.8962,0.0,4.0,4,4,78
MINDESTENS_5_MAXIMAL_95_PROZENT,2,B6 / B6S,true,0.9848,0.0,35.0,35,35,78



4) TOP-KOMPONENTEN DER k=6-REFERENZMODELLE
----------------------------------------------------------------------


variante,k,muster,rang_im_muster,komponente,gewicht,anteil_am_muster,produkte_mit_komponente,anteil_produkte
RAW,6,1,1,GJL1204417P0001,2.2919,0.0593,226,0.2927
RAW,6,1,2,GJL1204418P0001,2.2874,0.0592,220,0.285
RAW,6,1,3,GJL1206610P0008,2.2407,0.058,219,0.2837
RAW,6,1,4,2CDN361181P0111,1.9054,0.0493,557,0.7215
RAW,6,1,5,GJL1204204P0003,1.8386,0.0476,617,0.7992
RAW,6,1,6,GJL1203101P0001,1.8386,0.0476,617,0.7992
RAW,6,1,7,2CDN368502P0006,1.7817,0.0461,742,0.9611
RAW,6,1,8,GJL1201332R0002,1.7286,0.0448,756,0.9793
RAW,6,1,9,GJL1201331R0007,1.7286,0.0448,756,0.9793
RAW,6,1,10,GJL1201331R0008,1.7286,0.0448,756,0.9793



INTERPRETATIONSREGELN
- Fehler nur innerhalb derselben Datenvariante vergleichen.
- H-Stabilität nahe 1 ist gut; niedrige Werte bedeuten startabhängige Muster.
- Hohe Ähnlichkeit zweier H-Muster deutet auf redundante Muster hin.
- Konstant-1-Komponenten dürfen entfernt werden; selten/häufig nur bei robusterem Ergebnis.
- Anker-Ränge sollen über k, Seeds und vertretbare Filter stabil bleiben.
- Hohe Gruppenähnlichkeit ist weiterhin keine Wahrscheinlichkeit fachlicher Richtigkeit.

Fertig. 9 CSV-Dateien gespeichert unter: /Workspace/Users/khalil-said.albert@de.abb.com/nmf_evaluation_outputs
ZIP zum Weitergeben: /Workspace/Users/khalil-said.albert@de.abb.com/nmf_evaluation_outputs.zip


In [0]:
# ================================================================
# Schritt 5: Erster NMF-Pilot nur für Hauptgruppe GJL121
# ================================================================
import time
import numpy as np
import pandas as pd
from sklearn.decomposition import NMF

ZIEL_HAUPTGRUPPE = "GJL121"

# Produkte der ausgewählten Hauptgruppe bestimmen
ziel_produkte = produkt_info_alle.index[
    produkt_info_alle["hauptgruppe"] == ZIEL_HAUPTGRUPPE
]

# Nur ihre Produkt-Komponenten-Zeilen auswählen
X_ziel = X_produkte_alle.loc[ziel_produkte].copy()

# Komponenten entfernen, die in dieser Hauptgruppe kein einziges Mal
# vorkommen. Diese Spalten bestehen nur aus Nullen und enthalten
# für dieses Modell keine Information.
X_ziel = X_ziel.loc[:, X_ziel.sum(axis=0) > 0]

# Gruppenzuordnungen werden separat aufbewahrt.
# Sie gehen NICHT in das Modell ein.
ziel_info = produkt_info_alle.loc[X_ziel.index].copy()

print("Pilot-Hauptgruppe:", ZIEL_HAUPTGRUPPE)
print("Produkte im Modell:", X_ziel.shape[0])
print("Komponenten im Modell:", X_ziel.shape[1])
print(
    "Kleinere Gruppen vorhanden:",
    ziel_info["kleinere_gruppe"].nunique()
)

# Wir wissen noch nicht, wie viele Komponentenmustern sinnvoll sind.
muster_anzahlen = [2, 3, 4, 5, 6, 8, 10, 12, 15]

nmf_modelle = {}
nmf_ergebnisse = []

for anzahl_muster in muster_anzahlen:
    start = time.time()

    modell = NMF(
        n_components=anzahl_muster,
        init="nndsvda",
        random_state=20260908,
        max_iter=1000
    )

    # Hier beginnt das eigentliche Lernen.
    W = modell.fit_transform(X_ziel.values)
    H = modell.components_

    rekonstruktion = W @ H

    relativer_fehler = (
        np.linalg.norm(X_ziel.values - rekonstruktion)
        / np.linalg.norm(X_ziel.values)
    )

    dauer = time.time() - start

    nmf_modelle[anzahl_muster] = {
        "modell": modell,
        "W": W,
        "H": H,
        "produkte": list(X_ziel.index),
        "komponenten": list(X_ziel.columns),
    }

    nmf_ergebnisse.append({
        "anzahl_muster": anzahl_muster,
        "relativer_fehler": relativer_fehler,
        "durchlaeufe": int(modell.n_iter_),
        "dauer_sekunden": dauer,
    })

    print(
        f"{anzahl_muster:>2} Muster | "
        f"Fehler {relativer_fehler:.4f} | "
        f"{modell.n_iter_:>4} Durchläufe | "
        f"{dauer:.2f} Sekunden"
    )

nmf_ergebnisse = pd.DataFrame(nmf_ergebnisse)

print("\nPilotläufe abgeschlossen:", len(nmf_ergebnisse))
print("B6/B6S wurde beim Lernen nicht verwendet.")

Pilot-Hauptgruppe: GJL121
Produkte im Modell: 772
Komponenten im Modell: 103
Kleinere Gruppen vorhanden: 13
 2 Muster | Fehler 0.4760 |  170 Durchläufe | 0.04 Sekunden
 3 Muster | Fehler 0.4284 |   69 Durchläufe | 0.02 Sekunden
 4 Muster | Fehler 0.3934 |  112 Durchläufe | 0.03 Sekunden
 5 Muster | Fehler 0.3545 |   70 Durchläufe | 0.03 Sekunden
 6 Muster | Fehler 0.3341 |  203 Durchläufe | 0.09 Sekunden
 8 Muster | Fehler 0.3035 |  194 Durchläufe | 0.07 Sekunden


/databricks/python/lib/python3.10/site-packages/sklearn/decomposition/_nmf.py:1692: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


10 Muster | Fehler 0.2649 | 1000 Durchläufe | 0.33 Sekunden
12 Muster | Fehler 0.2453 |  244 Durchläufe | 0.33 Sekunden
15 Muster | Fehler 0.2110 |  400 Durchläufe | 0.48 Sekunden

Pilotläufe abgeschlossen: 9
B6/B6S wurde beim Lernen nicht verwendet.


In [0]:
# ================================================================
# Schritt 6: Gelernte Komponentenmustern sichtbar machen
# ================================================================

GEZEIGTE_MUSTERANZAHL = 6
TOP_KOMPONENTEN = 10

ansicht = nmf_modelle[GEZEIGTE_MUSTERANZAHL]
H = ansicht["H"]
komponenten_reihenfolge = ansicht["komponenten"]

# Häufigsten vorhandenen Kurztext je Komponente bestimmen.
# Der Kurztext wird ausschließlich angezeigt.
# Er wurde nicht zum Trainieren des Modells verwendet.
komponenten_texte = pd.DataFrame([
    {
        "komponente": str(zeile["komponente"]),
        "kurztext": str(zeile["komponente_txt"] or "").strip()
    }
    for zeile in sauber
])

komponenten_texte = komponenten_texte[
    komponenten_texte["kurztext"] != ""
]

text_haeufigkeit = (
    komponenten_texte
    .groupby(["komponente", "kurztext"])
    .size()
    .reset_index(name="vorkommen")
    .sort_values(
        ["komponente", "vorkommen"],
        ascending=[True, False]
    )
    .drop_duplicates("komponente")
    .set_index("komponente")["kurztext"]
    .to_dict()
)

muster_anzeige = []

for muster_index in range(H.shape[0]):
    gewichte = H[muster_index]

    # Komponenten mit dem höchsten Gewicht in diesem Muster
    top_positionen = np.argsort(gewichte)[::-1][:TOP_KOMPONENTEN]

    hoechstes_gewicht = gewichte[top_positionen[0]]

    for rang, position in enumerate(top_positionen, start=1):
        komponente = komponenten_reihenfolge[position]
        gewicht = gewichte[position]

        muster_anzeige.append({
            "muster": f"Muster {muster_index + 1}",
            "rang_im_muster": rang,
            "komponente": komponente,
            "kurztext_nur_anzeige":
                text_haeufigkeit.get(komponente, ""),
            "gewicht": gewicht,
            "relativ_zum_staerksten":
                gewicht / hoechstes_gewicht
                if hoechstes_gewicht > 0 else 0.0
        })

muster_anzeige = pd.DataFrame(muster_anzeige)

display(
    muster_anzeige.sort_values(
        ["muster", "rang_im_muster"]
    )
)

muster,rang_im_muster,komponente,kurztext_nur_anzeige,gewicht,relativ_zum_staerksten
Muster 1,1,GJL1204402P0018,Spulenanschlusswinkel (gerade),7.1122658780630985,1.0
Muster 1,2,GJL1201008R0001,"ANKER KPL. 0,25/0,22 BG BLAU CHROM.",4.97228959362589,0.6991146954956085
Muster 1,3,GJL1201005R0004,"JOCHSCHENKEL,GESCHWEISST BLAU CHROM.",4.97228959362589,0.6991146954956085
Muster 1,4,GJL1206623P0015,SPULENKOERPER Vampamid RAL9005 sw,4.97228959362589,0.6991146954956085
Muster 1,5,GJL1203609P0001,"RUECKSTELLFEDER 0,4N",3.8595980833845016,0.5426678571296037
Muster 1,6,GJL1204204P0003,JOCHSCHENKEL,3.325014090024761,0.46750418882403627
Muster 1,7,GJL1203101P0001,JOCH,3.325014090024761,0.46750418882403627
Muster 1,8,GJL1206603P0010,Kontakttraeger,2.1847485818901426,0.30718038658098096
Muster 1,9,GJL1203601P0002,"KONTAKTDRUCKFEDER 0,4N",2.1847485818901426,0.30718038658098096
Muster 1,10,GJL1206301P0005,ZWISCHENPLATTE,2.1847485818901426,0.30718038658098096


In [0]:
# ================================================================
# Schritt 7: Ähnlichkeit der kleineren Gruppen im NMF-Modell
# ================================================================
from itertools import combinations
from sklearn.metrics.pairwise import cosine_similarity

GEZEIGTE_MUSTERANZAHL = 5
ansicht = nmf_modelle[GEZEIGTE_MUSTERANZAHL]

# W als lesbare Tabelle:
# eine Zeile je Produkt, eine Spalte je gelerntem Muster
W_produkte = pd.DataFrame(
    ansicht["W"],
    index=ansicht["produkte"],
    columns=[
        f"Muster_{i + 1}"
        for i in range(GEZEIGTE_MUSTERANZAHL)
    ]
)

# Musterwerte jedes Produkts relativieren.
# Dadurch zählt jedes Produkt gleich stark, unabhängig davon,
# wie groß seine absoluten NMF-Werte sind.
zeilensummen = W_produkte.sum(axis=1).replace(0, np.nan)

W_relativ = (
    W_produkte
    .div(zeilensummen, axis=0)
    .fillna(0.0)
)

# Kleinere Gruppe jedes Produkts ergänzen
W_mit_gruppe = W_relativ.copy()
W_mit_gruppe["kleinere_gruppe"] = (
    ziel_info.loc[W_relativ.index, "kleinere_gruppe"]
)

# Durchschnittliches Musterprofil je kleinerer Gruppe
gruppen_muster = (
    W_mit_gruppe
    .groupby("kleinere_gruppe")
    .mean()
)

# Produktzahl je kleinerer Gruppe
produkte_je_gruppe = (
    W_mit_gruppe["kleinere_gruppe"]
    .value_counts()
)

print(
    f"Gruppenprofile aus dem Modell mit "
    f"{GEZEIGTE_MUSTERANZAHL} Mustern:"
)

gruppen_muster_anzeige = gruppen_muster.copy()
gruppen_muster_anzeige.insert(
    0,
    "produkte_N",
    produkte_je_gruppe.loc[gruppen_muster.index]
)

display(
    gruppen_muster_anzeige
    .reset_index()
    .round(4)
)

# Ähnlichkeit zwischen allen kleineren Gruppen berechnen
gruppen_namen = list(gruppen_muster.index)

S_gruppen = cosine_similarity(gruppen_muster.values)

paar_ergebnisse = []

for i, j in combinations(range(len(gruppen_namen)), 2):
    gruppe_a = gruppen_namen[i]
    gruppe_b = gruppen_namen[j]

    n_a = int(produkte_je_gruppe[gruppe_a])
    n_b = int(produkte_je_gruppe[gruppe_b])

    paar_ergebnisse.append({
        "gruppe_a": gruppe_a,
        "gruppe_b": gruppe_b,
        "produkte_a": n_a,
        "produkte_b": n_b,
        "nmf_aehnlichkeit": float(S_gruppen[i, j]),
        "hinweis":
            "mindestens eine Gruppe hat weniger als 3 Produkte"
            if min(n_a, n_b) < 3
            else ""
    })

paar_ergebnisse = (
    pd.DataFrame(paar_ergebnisse)
    .sort_values(
        "nmf_aehnlichkeit",
        ascending=False
    )
    .reset_index(drop=True)
)

paar_ergebnisse.insert(
    0,
    "rang",
    np.arange(1, len(paar_ergebnisse) + 1)
)

print(
    f"\nAlle {len(paar_ergebnisse)} Gruppenpaare, "
    "nach NMF-Ähnlichkeit sortiert:"
)

display(
    paar_ergebnisse.round({
        "nmf_aehnlichkeit": 4
    })
)

Gruppenprofile aus dem Modell mit 5 Mustern:


kleinere_gruppe,produkte_N,Muster_1,Muster_2,Muster_3,Muster_4,Muster_5
B6,255,0.1857,0.2138,0.3476,0.023,0.2299
B6S,8,0.0783,0.1627,0.2641,0.0131,0.4817
B6SE,1,0.0,0.0,0.2495,0.4856,0.2649
BASISTEILE,6,0.2946,0.3333,0.0,0.0388,0.3333
BC6,310,0.2451,0.2645,0.0037,0.0501,0.4367
MC09.01,2,0.0,0.0301,0.1692,0.0609,0.7397
MC09.10,2,0.0,0.0313,0.1793,0.0177,0.7717
VB6,45,0.0738,0.0417,0.2767,0.5289,0.0789
VB6A,38,0.0373,0.0133,0.2569,0.6723,0.0202
VB6S,2,0.0,0.058,0.2476,0.5911,0.1032



Alle 78 Gruppenpaare, nach NMF-Ähnlichkeit sortiert:


rang,gruppe_a,gruppe_b,produkte_a,produkte_b,nmf_aehnlichkeit,hinweis
1,MC09.01,MC09.10,2,2,0.9983,mindestens eine Gruppe hat weniger als 3 Produkte
2,VB6A,VB6SA,38,3,0.9962,
3,VB6S,VB6SA,2,3,0.9912,mindestens eine Gruppe hat weniger als 3 Produkte
4,VB6,VB6S,45,2,0.9885,mindestens eine Gruppe hat weniger als 3 Produkte
5,VB6A,VB6S,38,2,0.9871,mindestens eine Gruppe hat weniger als 3 Produkte
6,VB6,VB6A,45,38,0.9843,
7,VB6,VB6SA,45,3,0.9768,
8,BASISTEILE,BC6,6,310,0.9718,
9,VBC6,VBC6A,70,30,0.9575,
10,B6SE,VB6S,1,2,0.9512,mindestens eine Gruppe hat weniger als 3 Produkte


In [0]:
n_muster = ansicht["W"].shape[1]
muster_namen = [
    f"Muster_{i + 1}"
    for i in range(n_muster)
]

W_beschriftet = pd.DataFrame(
    ansicht["W"],
    index=ansicht["produkte"],
    columns=muster_namen
)

H_beschriftet = pd.DataFrame(
    ansicht["H"],
    index=muster_namen,
    columns=ansicht["komponenten"]
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None
):
    print("W: Produkte × Muster")
    display(W_beschriftet.round(4))

    print("H: Muster × Komponenten")
    display(H_beschriftet.round(4))

W: Produkte × Muster


Muster_1,Muster_2,Muster_3,Muster_4,Muster_5
0.0,0.0,0.1878,0.0506,0.6467
0.0,0.0,0.1874,0.0486,0.6484
0.0,0.0,0.1871,0.0502,0.6474
0.0,0.0,0.1958,0.0,0.6879
0.0,0.0,0.1953,0.0,0.6883
0.0,0.0,0.1951,0.0,0.6884
0.0,0.0,0.0,0.0,0.1768
0.0,0.0,0.1852,0.0452,0.6525
0.0,0.0,0.1841,0.0458,0.6526
0.0,0.0,0.1841,0.0458,0.6526


H: Muster × Komponenten


2CDC102006M6801,2CDN361181P0011,2CDN361181P0111,2CDN361181P0112,2CDN368102P0001,2CDN368502P0006,2CDN368507P0001,2CDN691010R0100,GHE3209613P0002,GHE3209613P0003,GHN 315222P0260,GHN 770402P0199,GHN 773405P0010,GHN 775732P0006,GHN775743P0004,GJL1201005R0004,GJL1201008R0001,GJL1201331R0007,GJL1201331R0008,GJL1201332R0002,GJL1201343R0011,GJL1201343R0012,GJL1201343R0013,GJL1201343R1014,GJL1201343R1018,GJL1201343R2012,GJL1201343R2017,GJL1201343R3014,GJL1201343R3017,GJL1201343R8010,GJL1201343R8014,GJL1201343R8015,GJL1201344R0011,GJL1201344R0013,GJL1201344R0014,GJL1201344R0015,GJL1201344R0016,GJL1201344R0017,GJL1201344R1013,GJL1201344R1016,GJL1201344R5011,GJL1201344R5014,GJL1201344R7011,GJL1201344R7012,GJL1201344R8010,GJL1201344R8011,GJL1201346R0001,GJL1201347R0003,GJL1201347R0004,GJL1201401R0008,GJL1201510R5014,GJL1201510R7012,GJL1201530R0012,GJL1201530R0019,GJL1201530R2010,GJL1201709R0001,GJL1201801R0001,GJL1201802R0001,GJL1203101P0001,GJL1203301P0003,GJL1203601P0002,GJL1203604P0001,GJL1203605P0001,GJL1203606P0001,GJL1203607P0001,GJL1203609P0001,GJL1203611P0001,GJL1203613P0001,GJL1204204P0003,GJL1204402P0008,GJL1204402P0018,GJL1204417P0001,GJL1204418P0001,GJL1204419P0001,GJL1206301P0005,GJL1206601P0014,GJL1206601P0015,GJL1206601P0104,GJL1206602P0012,GJL1206602P0112,GJL1206603P0010,GJL1206605P0026,GJL1206605P0102,GJL1206609P0003,GJL1206610P0008,GJL1206610P0101,GJL1206618P0010,GJL1206623P0015,GJL1206624P0008,GJL1206624P0009,GJL1206626P0002,GJL1206627P0004,GJL1206642P0002,GJL1206650P0001,GJL1207105P0005,GJL1207105P0006,GJL1207808P0004,GJL1207808P0005,GJL1209608P0006,GJL1209610P0001,GJL1209614P0005,GJR1928140P0002,GNGN368102P0011
0.0,0.0,2.4606,0.0,0.0,2.3748,2.4405,0.0,0.0236,0.0166,0.0,0.0,0.0,0.0,0.0,2.2248,2.2248,2.3373,2.3373,2.3373,0.0,0.0,0.0,0.0169,0.0016,0.0166,0.0148,0.0095,0.0205,0.0,0.0,0.0,0.1049,0.0763,0.0759,0.0744,0.0546,0.0788,0.0437,0.0523,0.0177,0.0,0.0,0.0,0.0,0.0711,0.0,0.0,0.0,0.0,0.0,0.0,0.0815,0.0507,0.0117,0.0,0.0,0.0,2.4763,1.4288,2.3162,0.4234,0.2241,0.0469,0.0122,1.5375,0.0,0.0112,2.4763,0.0,1.5099,2.983,2.9798,0.0,2.3162,0.0,2.3373,0.0,2.2081,0.0,2.3162,0.0,0.0,0.1721,2.917,0.0,0.0,2.2248,0.0464,0.0027,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0079,0.0,0.0,1.0E-4
0.0,0.0,0.0,2.748,0.0,2.4223,2.5602,0.0,0.053,0.0171,0.0,0.0113,0.0113,0.0113,0.0,2.3388,2.3388,2.4692,2.4692,2.4692,0.0087,0.0145,0.0,0.0174,0.0017,0.0171,0.0152,0.0035,0.0,0.0,0.008,0.0,0.089,0.0598,0.0758,0.0791,0.0384,0.0481,0.0446,0.0718,0.0774,0.0,0.0,0.0,0.0152,0.0738,0.0,0.0,0.0,0.0,0.0,0.0181,0.0691,0.0279,0.0114,0.0,0.0,0.0,2.6701,1.5945,2.4096,0.781,0.1198,0.0892,0.0,1.2466,0.0958,0.1119,2.6701,0.0493,1.4956,0.0,0.0,3.0187,2.4096,0.0,2.4692,0.0,2.5082,0.0,2.4096,0.0,0.0,0.0,0.0,0.0,3.0146,2.3388,0.0,0.0091,0.0,0.0,0.0,0.0113,2.8202,0.1435,0.0,0.0,0.0604,0.0,0.0493,0.0,0.0
0.0257,0.0194,0.4152,0.0,0.0194,0.7313,0.9514,0.273,0.6934,0.0917,0.0,0.3557,0.3557,0.3557,0.0526,1.3178,1.3178,0.935,0.935,0.935,0.3135,0.2703,0.2555,0.0908,0.047,0.0917,0.0983,0.1386,0.151,0.6792,0.3273,0.4202,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0638,0.0526,0.0,0.0,0.1779,0.1779,0.1779,0.0,0.017,0.0454,0.0,0.0,0.0,0.0257,0.0031,0.011,0.0,0.4953,1.1847,0.3827,0.0,0.0,0.007,0.9903,0.0,0.0,0.0,4.7457,0.0,0.2698,0.1495,0.0117,1.1847,0.0874,0.935,0.0905,0.6187,0.0905,1.1847,0.1233,0.0547,0.0,0.2405,0.0357,0.0212,1.3178,0.0,0.0,0.0,0.0,0.0,0.3557,0.0377,0.0,0.0079,0.0079,4.7067,2.338,4.7457,0.0,0.0175
0.0,0.0,1.4964,0.0,0.0,1.2305,0.3394,0.5649,0.0,0.0,0.0056,0.0,0.0,0.0,0.0,1.1329,1.1329,1.2546,1.2546,1.2546,0.0999,0.0637,0.0914,0.0,0.0,0.0,0.0,0.0,0.0,0.0246,0.0844,0.0425,0.122,0.0595,0.0574,0.0718,0.0267,0.0697,0.0,0.0333,0.1103,0.0,0.0202,0.0252,0.0,0.0833,0.0,0.0,0.0,0.9716,0.0,0.0106,0.0481,0.0628,0.0,0.0,0.0153,0.0233,0.1139,1.6647,1.2131,0.8473,0.0,0.1712,0.0,0.1566,0.0,0.1101,0.1139,0.0234,0.552,0.0,0.0273,0.5807,1.2131,0.0,1.2546,0.0,0.0,0.0,1.2131,0.0,0.0,1.129,0.0,0.0,0.5679,1.1329,2.1008,0.0504,0.9716,0.9716,0.9371,0.0,0.0,0.8217,0.008,0.008,0.023,0.0,0.0234,0.